# Treino QLoRA: Gemma-3-4B-it
Notebook para reproduzir o fluxo de treinamento com QLoRA no problema de analise de hallucination, usando o mesmo estilo do exemplo do artigo.

## 1. Dataset e conversao
Fluxo de preparacao de dados local, adaptado para analise de hallucination.

In [ ]:
from pathlib import Path

from src.schemas.dataset import BaseResultRow, JudgeResultRow
from src.dataset.judge_dataset import build_dataset, load_jsonl, select_records, stratified_split, to_conversation

results_dir = Path("data/results")
base_path = results_dir / "dataset_base.json"
judge_path = results_dir / "dataset_judge.jsonl"

base_results = load_jsonl(base_path, BaseResultRow)
judge_results = load_jsonl(judge_path, JudgeResultRow)

records = select_records(base_results, judge_results)

dataset = build_dataset(records)
dataset = stratified_split(dataset, validation_size=0.1, test_size=0.2, seed=42)

conversation_dataset = {split: to_conversation(dataset[split]) for split in ["train", "validation", "test"]}

Map:   0%|          | 0/720 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/720 [00:00<?, ? examples/s]

Map:   0%|          | 0/503 [00:00<?, ? examples/s]

Map:   0%|          | 0/72 [00:00<?, ? examples/s]

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

{'role': 'system', 'content': '\n# Task Description\nYou are an experienced code analyst, specializing in identifying and explaining possible code hallucinations.\n\nGiven the problem description and the user\'s code, you must analyze whether the code contains any hallucinations. The types of hallucinations are:\n\n- syntax_error: When the provided code has a syntax error in the Python language, such as incorrect or incomplete code.\n\n- runtime_error: When the code is "compilable" but has an error when executed, such as a call to an invalid function, an undefined variable, etc.\n\n- functional_error: When the code is executable, but it has some deviation from the correct functioning of what was requested.\n\n- correct: When the code is well-formed and has no functional errors.\n\nAssume that the code has a sequential line, that is, if it is at a lower level (such as syntax), even if it has a high level (such as functional_error), it should only be classified as a syntax error.\n\n# Pr

## 2. Modelo e processor
Carregar o Gemma-3-4B-it com o mesmo padrao do exemplo, usando AutoProcessor.

In [2]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

model_id = "google/gemma-3-4b-it"

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    dtype = torch.bfloat16
else:
    dtype = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_quant_storage=dtype,
)

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    dtype=dtype,
    device_map="auto",
    quantization_config=quantization_config,
)
processor = AutoProcessor.from_pretrained(model_id)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/mnt/f/Faculdade/HaluCodeDetection/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


## 3. LoRA e treino
Configurar QLoRA e o SFTTrainer para o dataset de hallucination.

In [ ]:
from peft import LoraConfig
from trl.trainer.sft_config import SFTConfig
from trl.trainer.sft_trainer import SFTTrainer

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
    ensure_weight_tying=True,
)

args = SFTConfig(
    output_dir="gemma-hallucination-qlora",
    max_length=512,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    optim="adamw_torch_fused",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=5e-5,
    fp16=True if model.dtype == torch.float16 else False,
    bf16=True if model.dtype == torch.bfloat16 else False,
    max_grad_norm=0.3,
    lr_scheduler_type="constant",
    push_to_hub=False,
    report_to="tensorboard",
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": True,
    },
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=conversation_dataset["train"],
    eval_dataset=conversation_dataset["validation"],
    peft_config=peft_config,
    processing_class=processor,
)

trainer.train()
trainer.save_model()

Tokenizing train dataset:   0%|          | 0/503 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/72 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss
1,0.233375,0.277485
2,0.169207,0.245707
3,0.107319,0.270630


## 4. Limpeza, merge e recarga
Liberar memoria, fundir o adapter LoRA com o modelo base e recarregar o modelo salvo para testar como no fluxo de evaluation.

In [4]:
from peft import PeftModel

from src.models.gemma import GemmaHandler

# Liberar memoria do treino
del model
del trainer
torch.cuda.empty_cache()

output_dir = "gemma-hallucination-qlora"
merged_model_dir = "merged_model"

# Recarregar o modelo base e fundir o adapter treinado
base_model = AutoModelForImageTextToText.from_pretrained("google/gemma-3-4b-it", low_cpu_mem_usage=True)
peft_model = PeftModel.from_pretrained(base_model, output_dir)
merged_model = peft_model.merge_and_unload()
merged_model.save_pretrained(merged_model_dir, safe_serialization=True, max_shard_size="2GB")

# Salvar o processor junto com o modelo fundido
processor = AutoProcessor.from_pretrained(output_dir)
processor.save_pretrained(merged_model_dir)

model_id = merged_model_dir

# Recarregar o modelo como sera usado no evaluation
handler = GemmaHandler(merged_model_dir)

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

## 5. Teste por split
Gerar uma amostra do treino, da validacao e do teste para comparar entrada, resposta esperada e saida do modelo.

In [5]:
from typing import Any, cast


def show_split_example(split_name: str, index: int = 0) -> dict[str, Any]:
    example = dataset[split_name][index]
    messages = conversation_dataset[split_name][index]["messages"]
    system_message = messages[0]["content"]
    user_message = messages[1]["content"]
    target_answer = messages[2]["content"] if len(messages) > 2 else ""
    
    response = handler.analyze_hallucination(
        example["problem_description"],
        example["generated_code"],
        {"max_new_tokens": 256, "do_sample": False},
        0,
    )

    result = {
        "split": split_name,
        "index": index,
        "system_message": system_message,
        "user_message": user_message,
        "target_answer": target_answer,
        "model_answer": response,
    }
    print("=" * 80)
    print(f"Split: {split_name} | index: {index}")
    print("\nSYSTEM:\n", system_message)
    print("\nUSER:\n", user_message)
    print("\nTARGET:\n", target_answer)
    print("\nMODEL:\n", response)
    
    return result


train_result = show_split_example("train", 0)
validation_result = show_split_example("validation", 0)
test_result = show_split_example("test", 0)

handler.close()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

{
    "train_result": train_result,
    "validation_result": validation_result,
    "test_result": test_result,
}

Messages: [{'role': 'system', 'content': '\n# Task Description\nYou are an experienced code analyst, specializing in identifying and explaining possible code hallucinations.\n\nGiven the problem description and the user\'s code, you must analyze whether the code contains any hallucinations. The types of hallucinations are:\n\n- syntax_error: When the provided code has a syntax error in the Python language, such as incorrect or incomplete code.\n\n- runtime_error: When the code is "compilable" but has an error when executed, such as a call to an invalid function, an undefined variable, etc.\n\n- functional_error: When the code is executable, but it has some deviation from the correct functioning of what was requested.\n\n- correct: When the code is well-formed and has no functional errors.\n\nAssume that the code has a sequential line, that is, if it is at a lower level (such as syntax), even if it has a high level (such as functional_error), it should only be classified as a syntax err

{'train_result': {'split': 'train',
  'index': 0,
  'system_message': '\n# Task Description\nYou are an experienced code analyst, specializing in identifying and explaining possible code hallucinations.\n\nGiven the problem description and the user\'s code, you must analyze whether the code contains any hallucinations. The types of hallucinations are:\n\n- syntax_error: When the provided code has a syntax error in the Python language, such as incorrect or incomplete code.\n\n- runtime_error: When the code is "compilable" but has an error when executed, such as a call to an invalid function, an undefined variable, etc.\n\n- functional_error: When the code is executable, but it has some deviation from the correct functioning of what was requested.\n\n- correct: When the code is well-formed and has no functional errors.\n\nAssume that the code has a sequential line, that is, if it is at a lower level (such as syntax), even if it has a high level (such as functional_error), it should only 